In [21]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai
import os
import json
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from underthesea import word_tokenize
from tqdm import tqdm
import faiss
import pickle

## 1. Load data

In [22]:
# SETUP & DATA LOADING
csv_file = '../../data/all_recipes_final.csv'
data = pd.read_csv(csv_file)

In [23]:
# Tạo combined text: Title + Description
data['combined_text'] = data['title'] + '. ' + data['ingredients_normalized']

print(f"Loaded {len(data)} recipes.")

Loaded 10263 recipes.


In [24]:
data.head()

,title,type_of_food,link,description,ingredients,ingredients_normalized,step,note,num_of_ingredients,cook_time,num_of_people,calories,source,combined_text
0,Cách muối dưa hành truyền thống,Món Tết,https://vnexpress.net/doi-song-cooking-cach-mu...,Dưa hành muối là món ăn truyền thống ngày Tết ...,"['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọ...","{'tro bếp hoặc nước vo gọa', 'đường', 'cà rốt ...",['Bước 1: Chọn hành củ: Nên chọn hành củ ta bá...,[],5,45 phút,8-10 người,459 kcal,vnexpress,Cách muối dưa hành truyền thống. {'tro bếp hoặ...
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,https://vnexpress.net/doi-song-cooking-su-hao-...,Đĩa xào khô ráo với su hào giòn ngọt quyện với...,"['2 củ su hào non', '1 con mực khô', '1/2 củ c...","{'muối', 'mỡ lợn hoặc dầu ăn', 'đường', 'gia v...",['Bước 1: Chọn và sơ chế mực: Người dân làng g...,['Su hào xào mực cùng với canh măng mực là hai...,6,50 phút,4 - 5 người,1.162 kcal,vnexpress,Su hào xào mực - món cổ Tết Bát Tràng. {'muối'...
2,Canh măng ngày Tết cổ truyền Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-canh-ma...,"Măng ngấu vị, giòn ngon, móng giò hầm vừa độ s...","['800 gr măng khô', '2 móng giò lợn', 'Nước dù...","{'muối', 'móng giò lợn', 'nước vo gạo ngâm măn...","['Bước 1: Chọn măng khô: Theo lối cũ, người nộ...",['Nếu tận dụng nước luộc gà nấu canh măng thì ...,6,100 phút,8 - 10 người,4.930 kcal,vnexpress,"Canh măng ngày Tết cổ truyền Hà Nội. {'muối', ..."
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-gia-han...,Đây là món ăn cổ truyền thường thấy trong cỗ T...,"['2 bộ lòng mề gà', '100 gr lạc', '50 gr hạt đ...","{'bộ lòng mề gà', 'muối', 'mỡ lợn', 'gia vị: m...",['Bước 1: Chọn và sơ chế lạc: Chọn lạc khô chắ...,['Hạnh nhân xào (hay giả hạnh nhân) là món ăn ...,8,60 phút,4-5 người,1.112 kcal,vnexpress,Giả hạnh nhân - món ngon Tết xưa Hà Nội. {'bộ ...
4,Chả bì ớt xiêm xanh,Món Tết,https://vnexpress.net/doi-song-cooking-cha-bi-...,"Chả bì bóng đẹp, gói đều tay. Khi ăn vị ngọt m...","['500 gr giò sống', '300 gr bì lợn', '20 - 30 ...","{'muối', 'gừng để sơ chế bì', 'bì lợn', 'hành ...","['Bước 1: Chọn và sơ chế bì lợn, chuẩn bị giò ...",['Nên sơ chế kỹ bì lợn để chả được thơm. Tùy t...,6,60 phút,5-6 người,2.512 kcal,vnexpress,"Chả bì ớt xiêm xanh. {'muối', 'gừng để sơ chế ..."


## 2. Method 1: SBERT + FAISS (Bi-Encoder Retrieval)

**Model-based approach using deep learning embeddings**
- Sử dụng Vietnamese SBERT để tạo semantic embeddings
- FAISS index cho fast similarity search
- Scalable và efficient cho large-scale retrieval

### 2.1. Initialize SBERT Model

In [25]:
# Load Vietnamese SBERT model
# Options: 'keepitreal/vietnamese-sbert', 'VoVanPhuc/sup-SimCSE-VietNamese-phobert-base', 
#          'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

sbert_model_name = 'keepitreal/vietnamese-sbert'
print(f"Loading SBERT model: {sbert_model_name}")

sbert_model = SentenceTransformer(sbert_model_name)
print(f"Model loaded successfully! Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}")

Loading SBERT model: keepitreal/vietnamese-sbert
Model loaded successfully! Embedding dimension: 768


### 2.2. Encode All Recipes to Embeddings

In [26]:
# Encode all recipes (offline, only once)
print(f"Encoding {len(data)} recipes to embeddings...")

recipe_embeddings_sbert = sbert_model.encode(
    data['combined_text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # L2 normalize for cosine similarity
)

print(f"Embeddings shape: {recipe_embeddings_sbert.shape}")
print(f"Embedding dimension: {recipe_embeddings_sbert.shape[1]}")

Encoding 10263 recipes to embeddings...


Batches: 100%|██████████| 321/321 [19:19<00:00,  3.61s/it]

Embeddings shape: (10263, 768)
Embedding dimension: 768


### 2.3. Build FAISS Index

In [27]:
# Build FAISS index for fast similarity search
embedding_dim = recipe_embeddings_sbert.shape[1]

# Use IndexFlatIP for Inner Product (cosine similarity on normalized vectors)
faiss_index_sbert = faiss.IndexFlatIP(embedding_dim)

# Add all recipe embeddings to index
faiss_index_sbert.add(recipe_embeddings_sbert)

print(f"FAISS index built successfully!")
print(f"Total vectors in index: {faiss_index_sbert.ntotal}")

FAISS index built successfully!
Total vectors in index: 10263


### 2.4. Save SBERT Model, Embeddings, and FAISS Index

In [28]:
# Create directory for SBERT method
sbert_save_dir = '../Saved_models/SBERT_FAISS'
os.makedirs(sbert_save_dir, exist_ok=True)

# Save embeddings
embeddings_path = os.path.join(sbert_save_dir, 'recipe_embeddings.npy')
np.save(embeddings_path, recipe_embeddings_sbert)

# Save FAISS index
faiss_path = os.path.join(sbert_save_dir, 'faiss_index.bin')
faiss.write_index(faiss_index_sbert, faiss_path)

# Save model name for later loading
model_info = {
    'model_name': sbert_model_name,
    'embedding_dim': embedding_dim,
    'num_recipes': len(data),
    'normalize_embeddings': True
}
info_path = os.path.join(sbert_save_dir, 'model_info.json')
with open(info_path, 'w', encoding='utf-8') as f:
    json.dump(model_info, f, indent=2, ensure_ascii=False)

print(f"\nSaved successfully: {sbert_save_dir}")


Saved successfully: ../Saved_models/SBERT_FAISS


### 2.5. Test SBERT + FAISS with Sample Queries

In [29]:
def sbert_search(query, k=10, threshold=0.5):
    """Search recipes using SBERT + FAISS"""
    # Encode query
    query_embedding = sbert_model.encode(
        [query], 
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Search in FAISS index
    scores, indices = faiss_index_sbert.search(query_embedding, k)
    
    # Filter by threshold
    valid_mask = scores[0] >= threshold
    valid_indices = indices[0][valid_mask]
    valid_scores = scores[0][valid_mask]
    
    return valid_indices, valid_scores

In [30]:
# Test queries
test_queries = [
    "Thịt kho nước dừa",
    "Canh chua cá lóc",
    "Món chay thanh đạm",
    "Bánh ngọt cho bữa sáng"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-"*80)
    
    indices, scores = sbert_search(query, k=5, threshold=0.3)
    
    if len(indices) == 0:
        print("   No results found above threshold")
    else:
        for i, (idx, score) in enumerate(zip(indices, scores), 1):
            recipe = data.iloc[idx]
            print(f"   {i}. [{score:.3f}] {recipe['title']}")
            print(f"      {recipe['description'][:100]}...")
    print()


Query: 'Thịt kho nước dừa'
--------------------------------------------------------------------------------
   1. [0.756] Thịt heo kho cùi dừa
      Món thịt kho tàu hay thịt kho hột vịt quá quen thuộc với những bữa cơm gia đình rồi, nhưng đến hôm n...
   2. [0.715] Món cá lóc kho tộ cực dễ, thơm ngon đậm đà đưa cơm
      Trời bắt đầu se lạnh, bữa ăn gia đình với chén cơm nóng hổi, miếng cá kho thơm nồng thì không còn gì...
   3. [0.700] Thịt kho tàu miền Bắc ngon ăn Tết bằng nồi đơn
      Thịt kho tàu miền Bắc thơm ngon và có nét đặc trưng riêng biệt không lẫn vào đâu được. Hãy cùng Vào ...
   4. [0.688] Thịt kho dừa thơm ngon, ngọt béo, đậm đà đưa cơm
      Thịt kho dừa là món ăn vô cùng quen thuộc trong bữa cơm của nhiều gia đình vì vị đậm đà của thịt, ng...
   5. [0.687] Thịt kho Tàu miền Nam ngon ngọt thơm lừng, mềm ngon đậm vị
      Thịt kho Tàu là một trong những món kho từ lâu đã gắn liền với người Việt. Đối với mỗi vùng miền khá...


Query: 'Canh chua cá lóc'
----------------

## 3. Method 2: Hybrid TF-IDF + SBERT (Ensemble)

**Ensemble approach combining lexical and semantic matching**
- TF-IDF: Captures exact keyword matching
- SBERT: Captures semantic similarity
- Weighted combination: α × TF-IDF + (1-α) × SBERT

### 3.1. Load Pre-trained TF-IDF Model

In [31]:
# Load pre-trained TF-IDF from content-based method

tfidf_dir = '../Saved_models/TFIDF'

# Load TF-IDF vectorizer
tfidf_vectorizer_path = os.path.join(tfidf_dir, 'tfidf_vectorizer.pkl')
with open(tfidf_vectorizer_path, 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
print(f"Loaded TF-IDF vectorizer")
print(f"  - Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

# Load TF-IDF processed data (contains matrix and other info)
tfidf_data_path = os.path.join(tfidf_dir, 'tfidf_processed_data.pkl')
with open(tfidf_data_path, 'rb') as f:
    tfidf_data = pickle.load(f)

Loaded TF-IDF vectorizer
  - Vocabulary size: 5000


In [32]:
# Extract TF-IDF matrix
if 'tfidf_matrix' in tfidf_data:
    tfidf_matrix = tfidf_data['tfidf_matrix']
else:
    # If matrix is stored separately
    tfidf_matrix_path = os.path.join(tfidf_dir, 'tfidf_matrix.pkl')
    if os.path.exists(tfidf_matrix_path):
        with open(tfidf_matrix_path, 'rb') as f:
            tfidf_matrix = pickle.load(f)
    else:
        # Fallback: transform current data
        print("  - TF-IDF matrix not found, transforming current data...")
        tfidf_matrix = tfidf_vectorizer.transform(data['combined_text'])

print(f"Loaded TF-IDF matrix")
print(f"  - Matrix shape: {tfidf_matrix.shape}")
print(f"  - Matrix sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")

  - TF-IDF matrix not found, transforming current data...
Loaded TF-IDF matrix
  - Matrix shape: (10263, 5000)
  - Matrix sparsity: 99.09%


### 3.2. Save Hybrid Model Components

In [33]:
# Create directory for Hybrid method
hybrid_save_dir = '../Saved_models/Hybrid_TFIDF_SBERT'
os.makedirs(hybrid_save_dir, exist_ok=True)

# Save SBERT embeddings (new component)
hybrid_embeddings_path = os.path.join(hybrid_save_dir, 'sbert_embeddings.npy')
np.save(hybrid_embeddings_path, recipe_embeddings_sbert)
print(f"Saved SBERT embeddings to {hybrid_embeddings_path}")

Saved SBERT embeddings to ../Saved_models/Hybrid_TFIDF_SBERT\sbert_embeddings.npy


In [34]:
# Save model configuration (needed for evaluation phase)
hybrid_config = {
    'sbert_model_name': sbert_model_name,
    'tfidf_source': '../Saved_models/TFIDF',
    'alpha': 0.5,  # Default weight for ensemble
    'num_recipes': len(data),
    'method': 'Hybrid_TFIDF_SBERT',
    'description': 'Ensemble of TF-IDF (lexical) and SBERT (semantic) with weighted combination'
}
config_path = os.path.join(hybrid_save_dir, 'config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(hybrid_config, f, indent=2, ensure_ascii=False)

print(f"\nHybrid TF-IDF + SBERT method saved")
print(f"   Location: {hybrid_save_dir}")
print(f"   Components:")
print(f"      - SBERT embeddings (new)")
print(f"      - TF-IDF (reused from {tfidf_dir})")
print(f"      - Config with alpha={hybrid_config['alpha']}")


Hybrid TF-IDF + SBERT method saved
   Location: ../Saved_models/Hybrid_TFIDF_SBERT
   Components:
      - SBERT embeddings (new)
      - TF-IDF (reused from ../Saved_models/TFIDF)
      - Config with alpha=0.5


### 3.3. Test Hybrid Method with Sample Queries

In [35]:
def hybrid_search(query, k=10, alpha=0.5, threshold=0.3):
    """
    Hybrid search combining TF-IDF and SBERT
    
    Args:
        query: Search query
        k: Number of top results
        alpha: Weight for TF-IDF (1-alpha for SBERT)
        threshold: Minimum combined score threshold
    
    Returns:
        indices: Recipe indices
        scores: Combined scores
        tfidf_scores: Individual TF-IDF scores
        sbert_scores: Individual SBERT scores
    """
    # 1. TF-IDF scoring
    query_tfidf = tfidf_vectorizer.transform([query])
    tfidf_scores = linear_kernel(query_tfidf, tfidf_matrix).flatten()
    
    # 2. SBERT scoring
    query_embedding = sbert_model.encode(
        [query], 
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    sbert_scores = cosine_similarity(query_embedding, recipe_embeddings_sbert).flatten()
    
    # 3. Combine scores
    combined_scores = alpha * tfidf_scores + (1 - alpha) * sbert_scores
    
    # 4. Get top-k
    top_indices = np.argsort(combined_scores)[::-1][:k]
    
    # 5. Filter by threshold
    valid_mask = combined_scores[top_indices] >= threshold
    valid_indices = top_indices[valid_mask]
    
    return (valid_indices, 
            combined_scores[valid_indices],
            tfidf_scores[valid_indices],
            sbert_scores[valid_indices])

In [36]:
alpha = 0.5  # Equal weight for TF-IDF and SBERT

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-"*80)
    
    indices, combined_scores, tfidf_scores, sbert_scores = hybrid_search(
        query, k=5, alpha=alpha, threshold=0.2
    )
    
    if len(indices) == 0:
        print("   No results found above threshold")
    else:
        for i, (idx, comb_score, tfidf_score, sbert_score) in enumerate(
            zip(indices, combined_scores, tfidf_scores, sbert_scores), 1
        ):
            recipe = data.iloc[idx]
            print(f"   {i}. [Combined: {comb_score:.3f}] (TF-IDF: {tfidf_score:.3f}, SBERT: {sbert_score:.3f})")
            print(f"      {recipe['title']}")
            print(f"      {recipe['description'][:100]}...")
    print()


Query: 'Thịt kho nước dừa'
--------------------------------------------------------------------------------
   1. [Combined: 0.594] (TF-IDF: 0.432, SBERT: 0.756)
      Thịt heo kho cùi dừa
      Món thịt kho tàu hay thịt kho hột vịt quá quen thuộc với những bữa cơm gia đình rồi, nhưng đến hôm n...
   2. [Combined: 0.592] (TF-IDF: 0.495, SBERT: 0.688)
      Thịt kho dừa thơm ngon, ngọt béo, đậm đà đưa cơm
      Thịt kho dừa là món ăn vô cùng quen thuộc trong bữa cơm của nhiều gia đình vì vị đậm đà của thịt, ng...
   3. [Combined: 0.544] (TF-IDF: 0.401, SBERT: 0.687)
      Thịt kho Tàu miền Nam ngon ngọt thơm lừng, mềm ngon đậm vị
      Thịt kho Tàu là một trong những món kho từ lâu đã gắn liền với người Việt. Đối với mỗi vùng miền khá...
   4. [Combined: 0.538] (TF-IDF: 0.513, SBERT: 0.564)
      Thịt kho cơm dừa
      Những miếng thịt kho màu nâu cánh gián thấm đượm gia vị càng thơm và mềm hơn khi được kết hợp với nh...
   5. [Combined: 0.525] (TF-IDF: 0.474, SBERT: 0.575)
      Vịt k